# 01 - Biological Age SBI With BayesFlow

This notebook is the walk-through notebook. It assumes notebook `00` has already fitted and saved the empirical baseline simulator.

Flow:

1. Load the fitted empirical model.
2. Build prior, bioindicator model, observation model, and BayesFlow simulator.
3. Run prior predictive and dependence checks.
4. Build the adapter.
5. Train a point-estimation network.
6. Validate on synthetic data.
7. Check predictions on held-out real baseline rows.


## Setup

In [68]:
import os
os.environ.setdefault("KERAS_BACKEND", "jax")

from pathlib import Path
from datetime import datetime
import copy
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import keras
import bayesflow as bf

SRC_CANDIDATES = [Path("../src"), Path("biological_age_sbi/experiment/src")]
for src in SRC_CANDIDATES:
    if (src / "bioage_sbi").exists():
        sys.path.insert(0, str(src.resolve()))
        break
else:
    raise FileNotFoundError("Could not find biological_age_sbi/experiment/src")

from bioage_sbi import LATENT_KEYS, MODEL_NAME, find_baseline_data_path, load_empirical_model
from bioage_sbi.simulator import make_component_functions, sample_component_model

sns.set_theme(style="whitegrid")
np.set_printoptions(suppress=True)

SEED = 1234
EXAMPLE_SEED = SEED
PRIOR_PREDICTIVE_SEED = SEED + 1
TRAINING_SEED = SEED + 2
VALIDATION_SEED = SEED + 3
NETWORK_SEED = SEED + 4
SYNTHETIC_RECOVERY_SEED = SEED + 5

SIMULATOR_VARIANT = "copula_empirical_residuals_sbp_bmi_0_5_sbp_agebin_mean_0_5"
SIMULATOR_DESCRIPTION = "Conditional Gaussian copula enabled; empirical continuous residual bootstrap enabled; SBP-BMI effect scaled by 0.5; train-only SBP age-bin mean correction scaled by 0.5."
COPULA_ENABLED = True
CONTINUOUS_RESIDUAL_NOISE_SCALE = 1.0
OBSERVATION_NOISE_SCALE = 1.0
LATENT_STD_SCALE = 1.0
DIRECT_AGE_EFFECT_SCALE = 1.0
LATENT_AGE_EFFECT_SCALE = 1.0
LATENT_LOADING_SCALE = 1.0
CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP = True
SBP_BMI_EFFECT_SCALE = 0.5
SBP_AGE_BIN_MEAN_CORRECTION = True
SBP_AGE_BIN_MEAN_CORRECTION_SCALE = 0.5
AGE_BIN_CALIBRATION_SYNTHETIC_N = None  # None means match train size.
AGE_BIN_CALIBRATION_SEED = SEED + 10_000
AGE_BINS = np.arange(40, 101, 10)


def _scale_age_coefficients(model_group, scale):
    if scale == 1.0:
        return
    for spec in model_group.values():
        for idx, feature in enumerate(spec["features"]):
            if feature in {"age_z", "age_z2"}:
                spec["coefficients"][idx] = float(spec["coefficients"][idx] * scale)


def _scale_loading_dict(loadings_by_variable, scale):
    if scale == 1.0:
        return
    for loadings in loadings_by_variable.values():
        for key, value in loadings.items():
            loadings[key] = float(value * scale)


def apply_simulator_variant(model):
    active_model = copy.deepcopy(model)

    for spec in active_model["continuous_models"].values():
        spec["residual_std"] = float(spec["residual_std"] * CONTINUOUS_RESIDUAL_NOISE_SCALE)

    _scale_age_coefficients(active_model["continuous_models"], DIRECT_AGE_EFFECT_SCALE)
    _scale_age_coefficients(active_model["binary_models"], DIRECT_AGE_EFFECT_SCALE)

    continuous_noise = active_model["observation_model"]["continuous_noise_std"]
    for key, value in continuous_noise.items():
        continuous_noise[key] = float(value * OBSERVATION_NOISE_SCALE)

    dependence_model = active_model.get("dependence_model")
    if dependence_model is not None:
        dependence_model["enabled"] = bool(COPULA_ENABLED)

    latent_config = active_model.get("latent_factors", {})
    for spec in latent_config.get("factors", {}).values():
        spec["std"] = float(spec["std"] * LATENT_STD_SCALE)
        if "age_z" in spec.get("mean_terms", {}):
            spec["mean_terms"]["age_z"] = float(spec["mean_terms"]["age_z"] * LATENT_AGE_EFFECT_SCALE)

    _scale_loading_dict(latent_config.get("continuous_loadings", {}), LATENT_LOADING_SCALE)
    _scale_loading_dict(latent_config.get("binary_logit_loadings", {}), LATENT_LOADING_SCALE)

    continuous_calibration = active_model.get("calibration", {}).get("continuous")
    if continuous_calibration is not None:
        continuous_calibration["enabled"] = bool(CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP)
        continuous_calibration["empirical_residual_bootstrap_enabled"] = bool(CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP)
        continuous_calibration.setdefault("effect_scales", {})["sbp_bmi"] = float(SBP_BMI_EFFECT_SCALE)

    active_model["simulator_variant"] = {
        "name": SIMULATOR_VARIANT,
        "description": SIMULATOR_DESCRIPTION,
        "copula_enabled": bool(active_model.get("dependence_model", {}).get("enabled", False)),
        "continuous_residual_noise_scale": CONTINUOUS_RESIDUAL_NOISE_SCALE,
        "observation_noise_scale": OBSERVATION_NOISE_SCALE,
        "latent_std_scale": LATENT_STD_SCALE,
        "direct_age_effect_scale": DIRECT_AGE_EFFECT_SCALE,
        "latent_age_effect_scale": LATENT_AGE_EFFECT_SCALE,
        "latent_loading_scale": LATENT_LOADING_SCALE,
        "continuous_empirical_residual_bootstrap": bool(CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP),
        "sbp_bmi_effect_scale": SBP_BMI_EFFECT_SCALE,
        "sbp_age_bin_mean_correction": bool(SBP_AGE_BIN_MEAN_CORRECTION),
        "sbp_age_bin_mean_correction_scale": SBP_AGE_BIN_MEAN_CORRECTION_SCALE,
    }
    return active_model


def synthetic_observed_frame(model, num_samples, seed):
    simulated = sample_component_model(model, num_samples=num_samples, seed=seed)
    observed_columns = [model["observed_key_by_column"][col] for col in model["columns"]]
    rename = {model["observed_key_by_column"][col]: col for col in model["columns"]}
    return simulated[["biological_age", *observed_columns]].rename(columns=rename)


def calibrate_sbp_age_bin_mean(model, train_frame, seed):
    if not SBP_AGE_BIN_MEAN_CORRECTION or "sbp" not in model["columns"]:
        return model

    continuous_calibration = model.get("calibration", {}).get("continuous")
    if continuous_calibration is None:
        return model

    n_synthetic = len(train_frame) if AGE_BIN_CALIBRATION_SYNTHETIC_N is None else AGE_BIN_CALIBRATION_SYNTHETIC_N
    synthetic_train = synthetic_observed_frame(model, n_synthetic, seed=seed)
    real_train = train_frame[["biological_age", "sbp"]].copy()

    synthetic_bins = pd.cut(synthetic_train["biological_age"], bins=AGE_BINS, include_lowest=True)
    real_bins = pd.cut(real_train["biological_age"], bins=AGE_BINS, include_lowest=True)
    adjustments = []
    for interval in synthetic_bins.cat.categories:
        synthetic_mean = synthetic_train.loc[synthetic_bins == interval, "sbp"].mean()
        real_mean = real_train.loc[real_bins == interval, "sbp"].mean()
        if np.isfinite(synthetic_mean) and np.isfinite(real_mean):
            adjustment = (real_mean - synthetic_mean) * SBP_AGE_BIN_MEAN_CORRECTION_SCALE
        else:
            adjustment = 0.0
        adjustments.append(float(adjustment))

    age_adjustment = continuous_calibration.setdefault("age_bin_mean_adjustment", {})
    age_adjustment["enabled"] = True
    age_adjustment.setdefault("columns", {})["sbp"] = adjustments
    age_adjustment["source"] = "train_split_synthetic_minus_real_age_bin_mean"
    age_adjustment["scale"] = float(SBP_AGE_BIN_MEAN_CORRECTION_SCALE)
    age_adjustment["synthetic_n"] = int(n_synthetic)
    return model


def reset_random_seeds(seed=SEED):
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)


def build_simulator(seed):
    prior, bioindicator_model, observation_model = make_component_functions(empirical_model, seed=seed)
    simulator = bf.make_simulator([prior, bioindicator_model, observation_model])
    return prior, bioindicator_model, observation_model, simulator


reset_random_seeds(SEED)


## Load Fitted Empirical Model

If this cell fails, run notebook `00` first. The JSON contains the biological-age prior, conditional regressions, observation noise, and adapter statistics.


In [ ]:
DATA_PATH = find_baseline_data_path()
EXPERIMENT_DIR = DATA_PATH.parents[2]
PROCESSED_DIR = DATA_PATH.parent
MODEL_PATH = PROCESSED_DIR / f"empirical_bioage_model_{MODEL_NAME}.json"
TRAIN_PATH = PROCESSED_DIR / f"baseline_train_{MODEL_NAME}.csv"
HOLDOUT_PATH = PROCESSED_DIR / f"baseline_holdout_{MODEL_NAME}.csv"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Run notebook 00 first. Missing: {MODEL_PATH}")

base_empirical_model = load_empirical_model(MODEL_PATH)
train_frame = pd.read_csv(TRAIN_PATH)
holdout_frame = pd.read_csv(HOLDOUT_PATH)
empirical_model = apply_simulator_variant(base_empirical_model)
empirical_model = calibrate_sbp_age_bin_mean(empirical_model, train_frame, seed=AGE_BIN_CALIBRATION_SEED)

model_continuous_columns = list(base_empirical_model["continuous_columns"])
model_binary_columns = list(base_empirical_model["binary_columns"])
model_columns = list(base_empirical_model["columns"])
model_true_keys = [base_empirical_model["true_key_by_column"][col] for col in model_columns]
model_latent_keys = [f"latent_{name}" for name in base_empirical_model.get("latent_factors", {}).get("factors", {})]
model_observed_key_by_column = base_empirical_model["observed_key_by_column"]

active_residual_noise = pd.DataFrame(
    {
        name: {
            "base_residual_std": base_empirical_model["continuous_models"][name]["residual_std"],
            "active_residual_std": empirical_model["continuous_models"][name]["residual_std"],
        }
        for name in model_continuous_columns
    }
).T

active_latent_noise = pd.DataFrame(
    {
        name: {
            "base_latent_std": base_empirical_model["latent_factors"]["factors"][name]["std"],
            "active_latent_std": empirical_model["latent_factors"]["factors"][name]["std"],
            "base_age_z": base_empirical_model["latent_factors"]["factors"][name]["mean_terms"].get("age_z", np.nan),
            "active_age_z": empirical_model["latent_factors"]["factors"][name]["mean_terms"].get("age_z", np.nan),
        }
        for name in base_empirical_model["latent_factors"]["factors"]
    }
).T

dependence_model = empirical_model.get("dependence_model", {"enabled": False, "type": "none", "columns": []})
copula_summary = pd.DataFrame(
    [
        {
            "model_version": empirical_model.get("version"),
            "model_family": empirical_model.get("model_family"),
            "dependence_type": dependence_model.get("type", "none"),
            "copula_enabled": dependence_model.get("enabled", False),
            "copula_columns": len(dependence_model.get("columns", [])),
            "min_eigenvalue_before_repair": dependence_model.get("min_eigenvalue_before_repair", np.nan),
        }
    ]
)

if dependence_model.get("columns"):
    copula_correlation = pd.DataFrame(
        dependence_model["correlation"],
        index=dependence_model["columns"],
        columns=dependence_model["columns"],
    )
else:
    copula_correlation = pd.DataFrame()

empirical_model["description"], empirical_model["simulator_variant"], train_frame.shape, holdout_frame.shape, active_residual_noise, active_latent_noise, copula_summary, copula_correlation.round(2)


## Simulator Components

The simulator factorization is:

`biological_age -> latent risks + copula residuals -> true bioindicators -> observed bioindicators`

Interdependence is created in three places: a conditional Gaussian copula over marginal residuals, a small age-dependent latent risk layer, and sequential conditioning where variables such as SBP depend on BMI and CVD depends on SBP, hypertension, diabetes, and smoking.


In [ ]:
example_prior_fn, example_bioindicator_model, example_observation_model, _ = build_simulator(EXAMPLE_SEED)
prior, bioindicator_model, observation_model, simulator = build_simulator(PRIOR_PREDICTIVE_SEED)

example_prior = example_prior_fn()
example_true = example_bioindicator_model(**example_prior)
example_observed = example_observation_model(**{key: example_true[key] for key in model_true_keys})
example_latents = {key: example_true[key] for key in model_latent_keys if key in example_true}

example_prior, example_latents, example_true, example_observed


## Sample From The Simulator

In [ ]:
num_samples = 1000
samples = simulator.sample(num_samples)
keras.tree.map_structure(keras.ops.shape, samples)


## Prior Predictive Check

Compare simulator outputs to the training data used to fit the empirical model. This checks the simulator before any neural network is involved.


In [ ]:
observed_key_by_column = empirical_model["observed_key_by_column"]
simulated_observed = pd.DataFrame({"biological_age": samples["biological_age"].squeeze()})
for col in empirical_model["columns"]:
    simulated_observed[col] = samples[observed_key_by_column[col]].squeeze()

simulated_observed.head()


In [ ]:
plot_columns = ["biological_age", *model_continuous_columns[:2]]
fig, axes = plt.subplots(1, len(plot_columns), figsize=(5 * len(plot_columns), 4))

for ax, col in zip(np.ravel(axes), plot_columns):
    sns.kdeplot(train_frame[col], label="Baseline train", ax=ax)
    sns.kdeplot(simulated_observed[col], label="Simulator", ax=ax)
    ax.set_title(col)
    ax.legend()
    sns.despine(ax=ax)

plt.tight_layout()


In [ ]:
prevalence_compare = pd.DataFrame(
    {
        "baseline_train": train_frame[model_binary_columns].mean(),
        "simulator": simulated_observed[model_binary_columns].mean(),
    }
)

ax = prevalence_compare.plot(kind="bar", figsize=(9, 4))
ax.set_ylim(0, 1)
ax.set_ylabel("Prevalence")
ax.set_title("Binary indicator prevalence")
sns.despine(ax=ax)
prevalence_compare


## Dependence Check

The simulator should preserve at least the broad correlation structure. With the conditional Gaussian copula enabled, this check should improve before we expect real-world recovery to improve.


In [ ]:
ordered_columns = ["biological_age", *model_columns]
real_corr = train_frame[ordered_columns].corr()
sim_corr = simulated_observed[ordered_columns].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(real_corr, vmin=-1, vmax=1, cmap="vlag", ax=axes[0])
axes[0].set_title("Baseline train correlations")
sns.heatmap(sim_corr, vmin=-1, vmax=1, cmap="vlag", ax=axes[1])
axes[1].set_title("Simulator correlations")
plt.tight_layout()


## Adapter

The adapter does the standard BayesFlow data preparation:

- convert `float64` to `float32`
- drop latent and true simulator-only variables
- concatenate target into `inference_variables`
- concatenate observed indicators into `inference_conditions`
- standardize conditions with statistics fitted in notebook `00`


In [ ]:
condition_keys = empirical_model["adapter"]["condition_keys"]
condition_mean = np.asarray(empirical_model["adapter"]["mean"], dtype="float32")
condition_std = np.asarray(empirical_model["adapter"]["std"], dtype="float32")

adapter = (
    bf.adapters.Adapter()
    .convert_dtype("float64", "float32")
    .drop([*model_latent_keys, *model_true_keys])
    .concatenate(["biological_age"], into="inference_variables")
    .concatenate(condition_keys, into="inference_conditions")
    .standardize("inference_conditions", mean=condition_mean, std=condition_std)
)

adapter


In [ ]:
adapted = adapter(samples)
keras.tree.map_structure(keras.ops.shape, adapted)


## Point Estimation Training

This is an overfit diagnostic, not a normal performance run. The point is to check whether the network and optimizer can memorize one or two fixed batches. If the training loss does not collapse, the problem is likely capacity, architecture, adapter shape, target scaling, or gradient updates rather than simulator realism.


In [ ]:
APPROACH_NAME = "overfit_two_batch_memorization_check"

num_training_batches = 2
num_validation_sets = 0
batch_size = 128
epochs = 300

MLP_WIDTHS = (512, 512, 256)
MLP_NORM = "layer"
MLP_DROPOUT = 0.0
MLP_RESIDUAL = True

NETWORK_VARIANT = f"{APPROACH_NAME}_mlp_{'_'.join(str(width) for width in MLP_WIDTHS)}_n{num_training_batches * batch_size}_e{epochs}__{SIMULATOR_VARIANT}"
NETWORK_DESCRIPTION = "Overfit diagnostic on two fixed synthetic batches; no validation data and dropout disabled."

_, _, _, training_simulator = build_simulator(TRAINING_SEED)
_, _, _, validation_simulator = build_simulator(VALIDATION_SEED)

training_data = training_simulator.sample(num_training_batches * batch_size)
validation_data = None


In [ ]:
reset_random_seeds(NETWORK_SEED)

q_levels = np.linspace(0.1, 0.9, 5)

point_inference_network = bf.networks.ScoringRuleNetwork(
    scoring_rules=dict(
        mean=bf.scoring_rules.MeanScore(),
        quantiles=bf.scoring_rules.QuantileScore(q_levels),
    ),
    subnet="mlp",
    subnet_kwargs=dict(
        widths=MLP_WIDTHS,
        norm=MLP_NORM,
        dropout=MLP_DROPOUT,
        residual=MLP_RESIDUAL,
    ),
)

point_inference_workflow = bf.BasicWorkflow(
    simulator=training_simulator,
    adapter=adapter,
    inference_network=point_inference_network,
)


In [ ]:
fit_kwargs = dict(epochs=epochs, batch_size=batch_size)
if validation_data is not None:
    fit_kwargs["validation_data"] = validation_data

history = point_inference_workflow.fit_offline(training_data, **fit_kwargs)


In [ ]:
f = bf.diagnostics.loss(history)


## Save And Reload Checkpoint

In [ ]:
checkpoint_path = EXPERIMENT_DIR / "checkpoints" / f"{MODEL_NAME}_point_estimator.keras"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

keras.saving.save_model(point_inference_workflow.approximator, checkpoint_path)
loaded = keras.saving.load_model(checkpoint_path)
point_inference_workflow.approximator = loaded

checkpoint_path


## Synthetic Recovery Check

This is the clean validity check because the simulator gives known ground truth biological age.


In [ ]:
_, _, _, synthetic_recovery_simulator = build_simulator(SYNTHETIC_RECOVERY_SEED)
val_sims = synthetic_recovery_simulator.sample(500)
estimates_point = point_inference_workflow.approximator.estimate(conditions=val_sims)
keras.tree.map_structure(keras.ops.shape, estimates_point)


In [ ]:
marker_mapping = dict(quantiles="_", mean="*")
par_names = ["Biological Age"]

f = bf.diagnostics.plots.recovery_from_estimates(
    estimates=estimates_point,
    targets=val_sims,
    marker_mapping=marker_mapping,
    variable_names=par_names,
    markersize=5,
)

for ax in np.ravel(f.axes):
    ax.set_xlim(0, 150)
    ax.set_ylim(0, 150)


In [ ]:
def extract_bioage_estimates(estimates):
    bioage_estimates = estimates.get("biological_age", estimates.get("inference_variables"))
    mean = np.asarray(bioage_estimates["mean"]).squeeze()
    quantiles = np.asarray(bioage_estimates["quantiles"]).squeeze()
    return mean, quantiles


def prediction_metrics(actual, mean, quantiles=None):
    error = mean - actual
    metrics = {
        "correlation_r": np.corrcoef(actual, mean)[0, 1],
        "mae": np.mean(np.abs(error)),
        "rmse": np.sqrt(np.mean(error ** 2)),
        "mean_error": np.mean(error),
    }
    if quantiles is not None:
        metrics["mean_q10_q90_width"] = np.mean(quantiles[:, -1] - quantiles[:, 0])
        metrics["q10_q90_coverage"] = np.mean((actual >= quantiles[:, 0]) & (actual <= quantiles[:, -1]))
    return pd.Series(metrics)


synthetic_mean, synthetic_quantiles = extract_bioage_estimates(estimates_point)
synthetic_actual = val_sims["biological_age"].squeeze()
synthetic_error = synthetic_mean - synthetic_actual

synthetic_metrics = prediction_metrics(synthetic_actual, synthetic_mean, synthetic_quantiles)
synthetic_metrics


## Synthetic Diagnostics

These diagnostics inspect the systematic young/old bias before changing the simulator again.

In [ ]:
def age_binned_diagnostics(actual, mean, quantiles=None, bins=None):
    if bins is None:
        bins = np.arange(40, 101, 10)

    diagnostics = pd.DataFrame(
        {
            "actual": actual,
            "mean": mean,
            "error": mean - actual,
        }
    )
    if quantiles is not None:
        diagnostics["q10"] = quantiles[:, 0]
        diagnostics["q50"] = quantiles[:, 2]
        diagnostics["q90"] = quantiles[:, -1]
        diagnostics["q10_q90_width"] = diagnostics["q90"] - diagnostics["q10"]
        diagnostics["covered_q10_q90"] = (diagnostics["actual"] >= diagnostics["q10"]) & (
            diagnostics["actual"] <= diagnostics["q90"]
        )

    diagnostics["age_bin"] = pd.cut(diagnostics["actual"], bins=bins, include_lowest=True)
    grouped = diagnostics.groupby("age_bin", observed=True)
    summary = grouped.agg(
        n=("actual", "size"),
        actual_mean=("actual", "mean"),
        predicted_mean=("mean", "mean"),
        mean_error=("error", "mean"),
        mae=("error", lambda x: np.mean(np.abs(x))),
        rmse=("error", lambda x: np.sqrt(np.mean(x**2))),
    )
    if quantiles is not None:
        summary["mean_q10_q90_width"] = grouped["q10_q90_width"].mean()
        summary["q10_q90_coverage"] = grouped["covered_q10_q90"].mean()
    return summary


synthetic_binned_diagnostics = age_binned_diagnostics(
    synthetic_actual,
    synthetic_mean,
    synthetic_quantiles,
)
synthetic_binned_diagnostics


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].axhline(0, color="black", linestyle="--", alpha=0.8)
axes[0].plot(
    synthetic_binned_diagnostics["actual_mean"],
    synthetic_binned_diagnostics["mean_error"],
    marker="o",
)
axes[0].set_xlabel("Actual biological age bin mean")
axes[0].set_ylabel("Mean prediction error")
axes[0].set_title("Synthetic error by age bin")

axes[1].plot(
    synthetic_binned_diagnostics["actual_mean"],
    synthetic_binned_diagnostics["mean_q10_q90_width"],
    marker="o",
)
axes[1].set_xlabel("Actual biological age bin mean")
axes[1].set_ylabel("Mean q10-q90 width")
axes[1].set_title("Synthetic uncertainty width by age bin")

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()


## Train/Validation Distribution Check

This checks whether apparent bias could come from different age distributions in training, validation, and recovery samples. In overfit mode there is no validation set.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

sns.kdeplot(training_data["biological_age"].squeeze(), label="training simulations", ax=ax)
if validation_data is not None:
    sns.kdeplot(validation_data["biological_age"].squeeze(), label="validation simulations", ax=ax)
sns.kdeplot(synthetic_actual, label="synthetic recovery", ax=ax)
ax.set_title("Age distributions across simulated datasets")
ax.set_xlabel("Biological age")
ax.legend()
sns.despine(ax=ax)
plt.tight_layout()


## Indicator-Age Slope Check

This compares broad age signal in the simulator against baseline training data. If simulator slopes are too weak, a bigger network will not fix the information bottleneck.


In [ ]:
def indicator_age_slopes(frame, indicator_columns):
    slopes = {}
    age = frame["biological_age"].to_numpy()
    for col in indicator_columns:
        values = frame[col].to_numpy()
        slopes[col] = np.polyfit(age, values, deg=1)[0]
    return pd.Series(slopes, name="slope_per_age_year")


indicator_columns = [*model_continuous_columns, *model_binary_columns]
indicator_slope_compare = pd.concat(
    [
        indicator_age_slopes(train_frame, indicator_columns).rename("baseline_train"),
        indicator_age_slopes(simulated_observed, indicator_columns).rename("simulator"),
    ],
    axis=1,
)
indicator_slope_compare["sim_minus_baseline"] = (
    indicator_slope_compare["simulator"] - indicator_slope_compare["baseline_train"]
)
indicator_slope_compare


## Real-World Holdout Check

Now apply the simulator-trained network to held-out real baseline rows. This is not a proof of validity, but it shows whether the learned estimator transfers back to the data distribution used to fit the simulator.


In [ ]:
def frame_to_conditions(frame):
    conditions = {
        model_observed_key_by_column[col]: frame[col].to_numpy(dtype="float64").reshape(-1, 1)
        for col in model_columns
    }
    conditions["biological_age"] = frame["biological_age"].to_numpy(dtype="float64").reshape(-1, 1)
    return conditions

real_conditions = frame_to_conditions(holdout_frame)
actual_bioage = holdout_frame["biological_age"].to_numpy()

real_estimates = point_inference_workflow.approximator.estimate(conditions=real_conditions)
real_mean, real_quantiles = extract_bioage_estimates(real_estimates)

real_eval = pd.DataFrame(
    {
        "actual_biological_age": actual_bioage,
        "predicted_mean": real_mean,
        "predicted_q10": real_quantiles[:, 0],
        "predicted_q50": real_quantiles[:, 2],
        "predicted_q90": real_quantiles[:, -1],
    }
)
real_eval["error"] = real_eval["predicted_mean"] - real_eval["actual_biological_age"]

real_metrics = prediction_metrics(actual_bioage, real_mean, real_quantiles)
real_metrics["n"] = len(real_eval)
real_metrics


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(
    real_eval["actual_biological_age"],
    real_eval["predicted_mean"],
    alpha=0.25,
    s=12,
)
axes[0].plot([0, 150], [0, 150], color="black", linestyle="--", alpha=0.8)
axes[0].set_xlim(0, 150)
axes[0].set_ylim(0, 150)
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_xlabel("Dataset biological age")
axes[0].set_ylabel("Predicted biological age")
axes[0].set_title(f"Real-world holdout check (r={real_metrics['correlation_r']:.2f})")

axes[1].scatter(
    real_eval["actual_biological_age"],
    real_eval["error"],
    alpha=0.25,
    s=12,
)
axes[1].axhline(0, color="black", linestyle="--", alpha=0.8)
axes[1].set_xlabel("Dataset biological age")
axes[1].set_ylabel("Prediction error")
axes[1].set_title("Error versus dataset biological age")

for ax in axes:
    sns.despine(ax=ax)

plt.tight_layout()


In [ ]:
real_eval.head()


## Real-World Diagnostics

The same binned diagnostics on held-out baseline rows show whether synthetic recovery behavior transfers back to the real data.


In [ ]:
real_binned_diagnostics = age_binned_diagnostics(
    actual_bioage,
    real_mean,
    real_quantiles,
)
real_binned_diagnostics


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].axhline(0, color="black", linestyle="--", alpha=0.8)
axes[0].plot(
    real_binned_diagnostics["actual_mean"],
    real_binned_diagnostics["mean_error"],
    marker="o",
)
axes[0].set_xlabel("Actual biological age bin mean")
axes[0].set_ylabel("Mean prediction error")
axes[0].set_title("Real holdout error by age bin")

axes[1].plot(
    real_binned_diagnostics["actual_mean"],
    real_binned_diagnostics["q10_q90_coverage"],
    marker="o",
)
axes[1].axhline(0.8, color="black", linestyle="--", alpha=0.7)
axes[1].set_ylim(0, 1)
axes[1].set_xlabel("Actual biological age bin mean")
axes[1].set_ylabel("q10-q90 coverage")
axes[1].set_title("Real holdout interval coverage by age bin")

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()


## Learning Curve Diagnostic

This diagnostic trains fresh networks on increasing numbers of fixed simulated batches: 1, 2, 8, 32, 128, and the full large setting of 1024 batches.

The saved `r` values are Pearson correlations between actual biological age and predicted mean biological age:

- `synthetic_r`: correlation on held-out simulator draws, where the ground-truth biological age is known from the simulator.
- `real_r`: correlation on held-out baseline rows, comparing predicted biological age against the dataset's biological-age estimate.

`r` measures linear association, not calibration. A model can have high `r` while still systematically overestimating young ages or underestimating old ages, so keep MAE/RMSE and age-bin bias plots in mind.


In [ ]:
LEARNING_CURVE_APPROACH_NAME = "batch_count_learning_curve"
LEARNING_CURVE_BATCH_COUNTS = [1, 2, 8, 32, 128, 1024]
LEARNING_CURVE_EPOCHS_BY_BATCH_COUNT = {
    1: 300,
    2: 300,
    8: 200,
    32: 120,
    128: 80,
    1024: 40,
}
LEARNING_CURVE_BATCH_SIZE = 128
LEARNING_CURVE_SYNTHETIC_EVAL_N = 500
LEARNING_CURVE_SYNTHETIC_SEED = SEED + 50
LEARNING_CURVE_TRAINING_SEED_BASE = SEED + 100
LEARNING_CURVE_NETWORK_SEED_BASE = SEED + 200

learning_curve_results_dir = EXPERIMENT_DIR / "results" / "network_diagnostics"
learning_curve_results_dir.mkdir(parents=True, exist_ok=True)
learning_curve_path = learning_curve_results_dir / f"{LEARNING_CURVE_APPROACH_NAME}_{SIMULATOR_VARIANT}.csv"
learning_curve_loss_history_path = learning_curve_results_dir / f"{LEARNING_CURVE_APPROACH_NAME}_{SIMULATOR_VARIANT}_loss_history.csv"
learning_curve_figure_path = learning_curve_results_dir / f"{LEARNING_CURVE_APPROACH_NAME}_{SIMULATOR_VARIANT}.png"
learning_curve_loss_figure_path = learning_curve_results_dir / f"{LEARNING_CURVE_APPROACH_NAME}_{SIMULATOR_VARIANT}_loss_history.png"

_, _, _, learning_curve_synthetic_simulator = build_simulator(LEARNING_CURVE_SYNTHETIC_SEED)
learning_curve_synthetic_eval = learning_curve_synthetic_simulator.sample(LEARNING_CURVE_SYNTHETIC_EVAL_N)
learning_curve_synthetic_actual = learning_curve_synthetic_eval["biological_age"].squeeze()
learning_curve_real_conditions = frame_to_conditions(holdout_frame)
learning_curve_real_actual = holdout_frame["biological_age"].to_numpy()


def history_to_loss_frame(history, run_metadata):
    history_dict = getattr(history, "history", history)
    if isinstance(history_dict, pd.DataFrame):
        history_dict = history_dict.to_dict(orient="list")
    if not isinstance(history_dict, dict):
        return pd.DataFrame()

    max_len = max((len(values) for values in history_dict.values() if hasattr(values, "__len__")), default=0)
    rows = []
    for epoch_index in range(max_len):
        row = {**run_metadata, "epoch": epoch_index + 1}
        for key, values in history_dict.items():
            if hasattr(values, "__len__") and epoch_index < len(values):
                value = values[epoch_index]
                try:
                    row[key] = float(value)
                except (TypeError, ValueError):
                    row[key] = value
        rows.append(row)
    return pd.DataFrame(rows)


def final_training_loss_from_history(history):
    history_dict = getattr(history, "history", history)
    if isinstance(history_dict, pd.DataFrame):
        history_dict = history_dict.to_dict(orient="list")
    if not isinstance(history_dict, dict):
        return {
            "loss_key": "unavailable",
            "first_train_loss": np.nan,
            "final_train_loss": np.nan,
            "min_train_loss": np.nan,
        }

    candidate_keys = ["loss", "train_loss", "training_loss"]
    candidate_keys.extend(
        key for key in history_dict if key.endswith("loss") and not key.startswith("val")
    )
    loss_key = next((key for key in candidate_keys if key in history_dict), None)
    if loss_key is None:
        return {
            "loss_key": "unavailable",
            "first_train_loss": np.nan,
            "final_train_loss": np.nan,
            "min_train_loss": np.nan,
        }

    loss_values = np.asarray(history_dict[loss_key], dtype=float)
    return {
        "loss_key": loss_key,
        "first_train_loss": float(loss_values[0]),
        "final_train_loss": float(loss_values[-1]),
        "min_train_loss": float(np.nanmin(loss_values)),
    }


def make_learning_curve_workflow(training_seed, network_seed):
    _, _, _, curve_training_simulator = build_simulator(training_seed)
    reset_random_seeds(network_seed)
    keras.backend.clear_session()

    curve_network = bf.networks.ScoringRuleNetwork(
        scoring_rules=dict(
            mean=bf.scoring_rules.MeanScore(),
            quantiles=bf.scoring_rules.QuantileScore(q_levels),
        ),
        subnet="mlp",
        subnet_kwargs=dict(
            widths=MLP_WIDTHS,
            norm=MLP_NORM,
            dropout=0.0,
            residual=MLP_RESIDUAL,
        ),
    )
    curve_workflow = bf.BasicWorkflow(
        simulator=curve_training_simulator,
        adapter=adapter,
        inference_network=curve_network,
    )
    return curve_training_simulator, curve_workflow


learning_curve_rows = []
learning_curve_loss_history_rows = []
for run_index, batch_count in enumerate(LEARNING_CURVE_BATCH_COUNTS):
    training_seed = LEARNING_CURVE_TRAINING_SEED_BASE + run_index
    network_seed = LEARNING_CURVE_NETWORK_SEED_BASE + run_index
    curve_epochs = LEARNING_CURVE_EPOCHS_BY_BATCH_COUNT[batch_count]
    num_training_sims = batch_count * LEARNING_CURVE_BATCH_SIZE

    curve_training_simulator, curve_workflow = make_learning_curve_workflow(training_seed, network_seed)
    curve_training_data = curve_training_simulator.sample(num_training_sims)
    curve_history = curve_workflow.fit_offline(
        curve_training_data,
        epochs=curve_epochs,
        batch_size=LEARNING_CURVE_BATCH_SIZE,
    )

    synthetic_estimates = curve_workflow.approximator.estimate(conditions=learning_curve_synthetic_eval)
    synthetic_mean_curve, synthetic_quantiles_curve = extract_bioage_estimates(synthetic_estimates)
    synthetic_metrics_curve = prediction_metrics(
        learning_curve_synthetic_actual,
        synthetic_mean_curve,
        synthetic_quantiles_curve,
    )

    real_estimates_curve = curve_workflow.approximator.estimate(conditions=learning_curve_real_conditions)
    real_mean_curve, real_quantiles_curve = extract_bioage_estimates(real_estimates_curve)
    real_metrics_curve = prediction_metrics(
        learning_curve_real_actual,
        real_mean_curve,
        real_quantiles_curve,
    )

    run_metadata = {
        "approach_name": LEARNING_CURVE_APPROACH_NAME,
        "simulator_variant": SIMULATOR_VARIANT,
        "batch_count": batch_count,
        "batch_size": LEARNING_CURVE_BATCH_SIZE,
        "num_training_sims": num_training_sims,
        "epochs": curve_epochs,
        "training_seed": training_seed,
        "network_seed": network_seed,
    }
    curve_loss_history = history_to_loss_frame(curve_history, run_metadata)
    if not curve_loss_history.empty:
        learning_curve_loss_history_rows.append(curve_loss_history)
        pd.concat(learning_curve_loss_history_rows, ignore_index=True).to_csv(
            learning_curve_loss_history_path,
            index=False,
        )

    loss_metrics = final_training_loss_from_history(curve_history)
    row = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "approach_name": LEARNING_CURVE_APPROACH_NAME,
        "simulator_variant": SIMULATOR_VARIANT,
        "model_version": empirical_model.get("version"),
        "model_family": empirical_model.get("model_family"),
        "dependence_model_type": empirical_model.get("dependence_model", {}).get("type", "none"),
        "copula_enabled": empirical_model.get("dependence_model", {}).get("enabled", False),
        "batch_count": batch_count,
        "batch_size": LEARNING_CURVE_BATCH_SIZE,
        "num_training_sims": num_training_sims,
        "epochs": curve_epochs,
        "mlp_widths": str(MLP_WIDTHS),
        "mlp_norm": MLP_NORM,
        "mlp_dropout": 0.0,
        "mlp_residual": MLP_RESIDUAL,
        "training_seed": training_seed,
        "network_seed": network_seed,
        **loss_metrics,
        "synthetic_r": float(synthetic_metrics_curve["correlation_r"]),
        "synthetic_mae": float(synthetic_metrics_curve["mae"]),
        "synthetic_rmse": float(synthetic_metrics_curve["rmse"]),
        "real_r": float(real_metrics_curve["correlation_r"]),
        "real_mae": float(real_metrics_curve["mae"]),
        "real_rmse": float(real_metrics_curve["rmse"]),
    }
    learning_curve_rows.append(row)

    learning_curve_results = pd.DataFrame(learning_curve_rows)
    learning_curve_results.to_csv(learning_curve_path, index=False)
    display(learning_curve_results.tail(1))

learning_curve_results = pd.DataFrame(learning_curve_rows)
learning_curve_results.to_csv(learning_curve_path, index=False)
if learning_curve_loss_history_rows:
    learning_curve_loss_history = pd.concat(learning_curve_loss_history_rows, ignore_index=True)
else:
    learning_curve_loss_history = pd.DataFrame()
learning_curve_loss_history.to_csv(learning_curve_loss_history_path, index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(
    learning_curve_results["num_training_sims"],
    learning_curve_results["final_train_loss"],
    marker="o",
    label="final training loss",
)
axes[0].plot(
    learning_curve_results["num_training_sims"],
    learning_curve_results["min_train_loss"],
    marker="o",
    linestyle="--",
    label="minimum training loss",
)
axes[0].set_xscale("log")
if (learning_curve_results[["final_train_loss", "min_train_loss"]] > 0).all().all():
    axes[0].set_yscale("log")
axes[0].set_xlabel("Training simulations")
axes[0].set_ylabel("Training loss")
axes[0].set_title("Training loss versus data size")
axes[0].legend()

axes[1].plot(
    learning_curve_results["num_training_sims"],
    learning_curve_results["synthetic_r"],
    marker="o",
    label="synthetic r",
)
axes[1].plot(
    learning_curve_results["num_training_sims"],
    learning_curve_results["real_r"],
    marker="o",
    label="real r",
)
axes[1].set_xscale("log")
axes[1].set_ylim(-0.05, 1.0)
axes[1].set_xlabel("Training simulations")
axes[1].set_ylabel("Pearson r")
axes[1].set_title("Recovery correlation versus data size")
axes[1].legend()

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()
fig.savefig(learning_curve_figure_path, dpi=160, bbox_inches="tight")

if not learning_curve_loss_history.empty:
    loss_key_for_plot = learning_curve_results["loss_key"].dropna().iloc[0]
    fig_loss, ax_loss = plt.subplots(figsize=(8, 5))
    for batch_count, group in learning_curve_loss_history.groupby("batch_count"):
        if loss_key_for_plot in group:
            ax_loss.plot(
                group["epoch"],
                group[loss_key_for_plot],
                label=f"{batch_count} batches ({int(group['num_training_sims'].iloc[0])} sims)",
            )
    if loss_key_for_plot in learning_curve_loss_history and (learning_curve_loss_history[loss_key_for_plot] > 0).all():
        ax_loss.set_yscale("log")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel(loss_key_for_plot)
    ax_loss.set_title("Training loss convergence by data size")
    ax_loss.legend(fontsize=8)
    sns.despine(ax=ax_loss)
    plt.tight_layout()
    fig_loss.savefig(learning_curve_loss_figure_path, dpi=160, bbox_inches="tight")

print(f"Saved learning-curve CSV to: {learning_curve_path}")
print(f"Saved learning-curve loss history CSV to: {learning_curve_loss_history_path}")
print(f"Saved learning-curve figure to: {learning_curve_figure_path}")
print(f"Saved learning-curve loss figure to: {learning_curve_loss_figure_path}")
learning_curve_results.round(3)


## Save Run History

This appends one compact row per completed run. Update `NETWORK_DESCRIPTION` when you change the approach.

In [ ]:
RESULTS_DIR = EXPERIMENT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_PATH = RESULTS_DIR / "bioage_point_estimation_history.csv"

history_record = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "approach_name": APPROACH_NAME,
    "run_label": NETWORK_VARIANT,
    "description": NETWORK_DESCRIPTION,
    "model_name": MODEL_NAME,
    "model_version": empirical_model.get("version"),
    "model_family": empirical_model.get("model_family"),
    "dependence_model_type": empirical_model.get("dependence_model", {}).get("type", "none"),
    "copula_enabled": empirical_model.get("dependence_model", {}).get("enabled", False),
    "simulator_variant": SIMULATOR_VARIANT,
    "simulator_description": SIMULATOR_DESCRIPTION,
    "continuous_residual_noise_scale": CONTINUOUS_RESIDUAL_NOISE_SCALE,
    "observation_noise_scale": OBSERVATION_NOISE_SCALE,
    "latent_std_scale": LATENT_STD_SCALE,
    "direct_age_effect_scale": DIRECT_AGE_EFFECT_SCALE,
    "latent_age_effect_scale": LATENT_AGE_EFFECT_SCALE,
    "latent_loading_scale": LATENT_LOADING_SCALE,
    "continuous_empirical_residual_bootstrap": CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP,
    "sbp_bmi_effect_scale": SBP_BMI_EFFECT_SCALE,
    "sbp_age_bin_mean_correction": SBP_AGE_BIN_MEAN_CORRECTION,
    "sbp_age_bin_mean_correction_scale": SBP_AGE_BIN_MEAN_CORRECTION_SCALE,
    "seed": SEED,
    "training_seed": TRAINING_SEED,
    "validation_seed": VALIDATION_SEED,
    "network_seed": NETWORK_SEED,
    "synthetic_recovery_seed": SYNTHETIC_RECOVERY_SEED,
    "num_training_batches": num_training_batches,
    "batch_size": batch_size,
    "num_training_sims": num_training_batches * batch_size,
    "num_validation_sets": num_validation_sets,
    "epochs": epochs,
    "mlp_widths": str(MLP_WIDTHS),
    "mlp_norm": MLP_NORM,
    "mlp_dropout": MLP_DROPOUT,
    "mlp_residual": MLP_RESIDUAL,
    "synthetic_r": synthetic_metrics["correlation_r"],
    "synthetic_mae": synthetic_metrics["mae"],
    "synthetic_rmse": synthetic_metrics["rmse"],
    "synthetic_mean_error": synthetic_metrics["mean_error"],
    "synthetic_q10_q90_coverage": synthetic_metrics["q10_q90_coverage"],
    "synthetic_mean_q10_q90_width": synthetic_metrics["mean_q10_q90_width"],
    "real_r": real_metrics["correlation_r"],
    "real_mae": real_metrics["mae"],
    "real_rmse": real_metrics["rmse"],
    "real_mean_error": real_metrics["mean_error"],
    "real_q10_q90_coverage": real_metrics["q10_q90_coverage"],
    "real_mean_q10_q90_width": real_metrics["mean_q10_q90_width"],
}

history_row = pd.DataFrame([history_record])
if HISTORY_PATH.exists():
    previous_history = pd.read_csv(HISTORY_PATH)
    run_history = pd.concat([previous_history, history_row], ignore_index=True)
else:
    run_history = history_row

run_history.to_csv(HISTORY_PATH, index=False)

history_display_columns = [
    "timestamp",
    "approach_name",
    "run_label",
    "model_version",
    "model_family",
    "dependence_model_type",
    "copula_enabled",
    "simulator_variant",
    "continuous_residual_noise_scale",
    "observation_noise_scale",
    "latent_std_scale",
    "direct_age_effect_scale",
    "latent_age_effect_scale",
    "latent_loading_scale",
    "continuous_empirical_residual_bootstrap",
    "sbp_bmi_effect_scale",
    "sbp_age_bin_mean_correction",
    "sbp_age_bin_mean_correction_scale",
    "num_training_sims",
    "num_validation_sets",
    "epochs",
    "mlp_widths",
    "mlp_dropout",
    "synthetic_r",
    "synthetic_mae",
    "synthetic_rmse",
    "synthetic_q10_q90_coverage",
    "synthetic_mean_q10_q90_width",
    "real_r",
    "real_mae",
    "real_rmse",
    "real_q10_q90_coverage",
    "real_mean_q10_q90_width",
]

history_display = run_history[[col for col in history_display_columns if col in run_history.columns]].copy()
number_columns = history_display.select_dtypes(include="number").columns
history_display[number_columns] = history_display[number_columns].round(3)

print(f"Saved run history to: {HISTORY_PATH}")
history_display
